# TP1 — Regression: Model Comparison (Diabetes)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp1_correction.ipynb)

## Objective

Apply the models learned in Session 2 (KNN, Decision Trees, SVM, Random Forests, Gradient Boosting) to a **regression** problem. Compare their performance and find the best model through hyperparameter tuning.

### Roadmap

1. **Quick EDA** — load and explore the Diabetes dataset
2. **Prepare the data** — train/test split + scaling
3. **Baseline** — Linear Regression as reference
4. **Try 5 different models** — with manual hyperparameter exploration
5. **Compare all models** — table and chart
6. **Systematic tuning** — GridSearchCV on the best model
7. **Final evaluation** — cross-validation

> **Link to Session 1:** In TP1 of Session 1, you built a Linear Regression on California Housing. Now you'll use that as a baseline and try to beat it with the non-parametric models from Session 2.

> **See the `tp_0_*.ipynb` notebooks** for detailed explanations of each algorithm.


In [ ]:
!pip install scikit-learn


---
## 1. Setup


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

---
## 2. Load & Explore the Data

Since you already mastered EDA in Session 1, we keep this section short. The goal is to quickly understand what we're working with.


### Exercise 1.1 — Load the dataset

Load the Diabetes dataset and display its shape, feature names, and first rows.

**Why:** Understand the dimensions and content of the data before modeling.

*Hint:* `load_diabetes(as_frame=True)`, `df.shape`, `df.head()`


In [2]:
data = load_diabetes(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}")
print(f"\nFeatures: {list(data.feature_names)}")
df.head()


Shape: (442, 11)

Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [10]:
df["bmi"] 

0      0.061696
1     -0.051474
2      0.044451
3     -0.011595
4     -0.036385
         ...   
437    0.019662
438   -0.015906
439   -0.015906
440    0.039062
441   -0.073030
Name: bmi, Length: 442, dtype: float64

These are the **10 features** of the Diabetes dataset:

- **age** — Age of the patient
- **sex** — Sex of the patient
- **bmi** — Body mass index
- **bp** — Average blood pressure
- **s1–s6** — Six blood serum measurements

All features have been **preprocessed**: centered and scaled to have zero mean and unit variance.

**Target:** A quantitative measure of **disease progression** one year after baseline.


### Exercise 1.2 — Summary statistics

Display summary statistics for each column.

**Why:** Understand the range and distribution of each feature.

*Hint:* `df.describe()`

**Question:** What do you notice about the feature scales? Are they similar or different?


In [11]:
df.describe()


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,442.000000
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.268604e-17,1.130318e-17,152.133484
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,77.093005
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01,25.000000
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02,87.000000
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03,140.500000
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02,211.500000
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01,346.000000


**Answer:** All features have been preprocessed: centered and scaled. They have similar ranges (roughly -0.2 to +0.2). This means feature scaling won't change much, but we'll apply it for good practice.


### Exercise 1.3 — Target distribution

Plot a histogram of the target variable.

**Why:** Understand the distribution of what we're predicting — is it symmetric? Are there outliers?

*Hint:* `plt.hist(df['target'], bins=30, edgecolor='k', alpha=0.7)`


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['target'], bins=30, edgecolor='k', alpha=0.7)
plt.xlabel('Disease Progression')
plt.ylabel('Count')
plt.title('Distribution of Target Variable')
plt.show()


---
## 3. Prepare the Data

We follow the same methodology as Session 1: **split first, then scale**. This prevents data leakage from the test set into the training process.


### Exercise 2.1 — Split and scale

Separate features and target, split into train/test (80/20), and apply StandardScaler.

**Why:** Proper evaluation requires unseen test data. Scaling ensures all features contribute equally.

*Hint:* `train_test_split(test_size=0.2, random_state=42)`, `StandardScaler()`, `fit_transform` on train, `transform` on test.


In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")


---
## 4. Baseline — Linear Regression

Before trying complex models, establish a **baseline**. Linear Regression from Session 1 is our reference point. Any new model should aim to beat this baseline.


### Exercise 3.1 — Train and evaluate baseline

Train a `LinearRegression`, evaluate MAE on both train and test sets. Store the test MAE as `baseline_mae` for later comparison.

**Why:** A baseline gives context to all subsequent results. Without it, you can't tell if a model is "good" or just "less bad."

*Hint:* `LinearRegression()`, `model.fit(...)`, `mean_absolute_error(...)`

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train_scaled, y_train)

y_pred_train = baseline.predict(X_train_scaled)
y_pred_test = baseline.predict(X_test_scaled)

print("=== Baseline: Linear Regression ===")
print(f"Train MAE: {mean_absolute_error(y_train, y_pred_train):.2f}")
print(f"Test  MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")

baseline_mae = mean_absolute_error(y_test, y_pred_test)

**Your goal:** Can you beat this baseline MAE with the models you learned in Session 2?

---
## 5. Explore Different Models

Now try each model from the `tp_0` notebooks. For each model, experiment with different hyperparameter values and record the best test MAE.

We'll store results in a dictionary for comparison at the end.

### Exercise 4.1 — KNN Regressor

K-Nearest Neighbors predicts by averaging the target values of the K closest training samples. See `tp_0_knn.ipynb` for details.

Try `k_values = [1, 3, 5, 10, 20]`. For each k, report train MAE and test MAE. Store the best test MAE in a `results` dictionary.

**Why:** KNN is a simple non-parametric model. The key hyperparameter `k` controls the bias-variance trade-off.

*Hint:* `KNeighborsRegressor(n_neighbors=k)`

**Question:** What happens as k increases? Which k gives the best test MAE?

In [ ]:
results = {'Linear Regression': baseline_mae}

print("=== KNN Regressor ===")
k_values = [1, 3, 5, 10, 20]
best_mae, best_k = np.inf, None

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, knn.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, knn.predict(X_test_scaled))
    print(f"  k={k:>2d}:  Train MAE = {train_mae:.2f},  Test MAE = {test_mae:.2f}")
    if test_mae < best_mae:
        best_mae, best_k = test_mae, k

results['KNN'] = best_mae
print(f"\nBest: k={best_k}, Test MAE = {best_mae:.2f}")

**Answer:** As k increases, the model becomes smoother (less overfitting). Small k overfits (low train MAE, higher test MAE). The best k balances bias and variance.

### Exercise 4.2 — Decision Tree Regressor

Decision Trees split the feature space into regions and predict the average target value in each region. See `tp_0_tree.ipynb` for details.

Try `depths = [2, 5, 10, None]`. For each depth, report train MAE and test MAE. Store the best test MAE in `results`.

**Why:** `max_depth` controls how complex the tree can be. Too deep = overfitting, too shallow = underfitting.

*Hint:* `DecisionTreeRegressor(max_depth=depth, random_state=42)`

**Question:** What happens with `max_depth=None`? Compare train and test MAE.

In [ ]:
print("=== Decision Tree Regressor ===")
depths = [2, 5, 10, None]
best_mae, best_depth = np.inf, None

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, tree.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, tree.predict(X_test_scaled))
    depth_str = str(depth) if depth is not None else "None"
    print(f"  max_depth={depth_str:>4s}:  Train MAE = {train_mae:.2f},  Test MAE = {test_mae:.2f}")
    if test_mae < best_mae:
        best_mae, best_depth = test_mae, depth

results['Decision Tree'] = best_mae
print(f"\nBest: max_depth={best_depth}, Test MAE = {best_mae:.2f}")

**Answer:** With `max_depth=None`, the tree overfits massively: near-zero train MAE but much higher test MAE. Limiting depth prevents overfitting.

### Exercise 4.3 — Support Vector Regressor (SVR)

SVR finds a hyperplane that fits the data within an epsilon-tube margin. See `tp_0_svm.ipynb` for details.

Try the following configurations:
```python
configs = [
    {'kernel': 'linear', 'C': 1.0},
    {'kernel': 'rbf', 'C': 0.1},
    {'kernel': 'rbf', 'C': 1.0},
    {'kernel': 'rbf', 'C': 10.0},
]
```

**Why:** The kernel determines how the model maps features to a higher-dimensional space. C controls the regularization strength.

*Hint:* `SVR(**cfg)`

**Question:** Which kernel works best? How does C affect performance?


In [ ]:
print("=== Support Vector Regressor ===")
configs = [
    {'kernel': 'linear', 'C': 1.0},
    {'kernel': 'rbf', 'C': 0.1},
    {'kernel': 'rbf', 'C': 1.0},
    {'kernel': 'rbf', 'C': 10.0},
]
best_mae, best_cfg = np.inf, None

for cfg in configs:
    svr = SVR(**cfg)
    svr.fit(X_train_scaled, y_train)
    test_mae = mean_absolute_error(y_test, svr.predict(X_test_scaled))
    print(f"  {cfg}: Test MAE = {test_mae:.2f}")
    if test_mae < best_mae:
        best_mae, best_cfg = test_mae, cfg

results['SVR'] = best_mae
print(f"\nBest: {best_cfg}, Test MAE = {best_mae:.2f}")

**Answer:** The RBF kernel typically performs better than linear as it can capture non-linear patterns. Higher C values make the model fit tighter to the data, but too high may overfit.


### Exercise 4.4 — Random Forest Regressor

Random Forests are ensembles of Decision Trees trained on random subsets of data and features. See `tp_0_bagging.ipynb` for details.

Try the following configurations:
```python
configs = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': None},
]
```

**Why:** `n_estimators` controls the number of trees. `max_depth` limits individual tree complexity.

*Hint:* `RandomForestRegressor(**cfg, random_state=42)`

**Question:** How does `n_estimators` affect performance? Does more trees always help?


In [ ]:
print("=== Random Forest Regressor ===")
configs = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': None},
]
best_mae, best_cfg = np.inf, None

for cfg in configs:
    rf = RandomForestRegressor(**cfg, random_state=42)
    rf.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, rf.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, rf.predict(X_test_scaled))
    print(f"  {cfg}: Train MAE = {train_mae:.2f}, Test MAE = {test_mae:.2f}")
    if test_mae < best_mae:
        best_mae, best_cfg = test_mae, cfg

results['Random Forest'] = best_mae
print(f"\nBest: {best_cfg}, Test MAE = {best_mae:.2f}")

**Answer:** More trees generally improve performance up to a point, then the gains saturate. Unlike single trees, Random Forests are resistant to overfitting from adding more trees. `max_depth` also matters to prevent individual trees from overfitting.


### Exercise 4.5 — Gradient Boosting Regressor

Gradient Boosting builds trees sequentially, each one correcting the errors of the previous ensemble. See `tp_0_boosting.ipynb` for details.

Try the following configurations:
```python
configs = [
    {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5},
]
```

**Why:** `learning_rate` controls how much each tree contributes. Smaller learning rates need more trees but often generalize better.

*Hint:* `GradientBoostingRegressor(**cfg, random_state=42)`

**Question:** What is the effect of `learning_rate`? How does it interact with `n_estimators`?


In [ ]:
print("=== Gradient Boosting Regressor ===")
configs = [
    {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 5},
]
best_mae, best_cfg = np.inf, None

for cfg in configs:
    gbr = GradientBoostingRegressor(**cfg, random_state=42)
    gbr.fit(X_train_scaled, y_train)
    train_mae = mean_absolute_error(y_train, gbr.predict(X_train_scaled))
    test_mae = mean_absolute_error(y_test, gbr.predict(X_test_scaled))
    print(f"  lr={cfg['learning_rate']}, n={cfg['n_estimators']}, d={cfg['max_depth']}: Train MAE = {train_mae:.2f}, Test MAE = {test_mae:.2f}")
    if test_mae < best_mae:
        best_mae, best_cfg = test_mae, cfg

results['Gradient Boosting'] = best_mae
print(f"\nBest: {best_cfg}, Test MAE = {best_mae:.2f}")

**Answer:** A smaller `learning_rate` requires more estimators to converge, but often generalizes better. There is a trade-off: small learning rate + many trees = better generalization but slower training.


---
## 6. Model Comparison

Now let's compare all models side-by-side to see which performed best.


### Exercise 5.1 — Build comparison table

Create a DataFrame from the `results` dictionary, sorted by MAE in ascending order.

**Why:** A structured comparison makes it easy to identify the best model at a glance.

*Hint:* `pd.DataFrame(...)`, `.sort_values('Test MAE', ascending=True)`

In [ ]:
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Test MAE': list(results.values())
}).sort_values('Test MAE', ascending=True).reset_index(drop=True)

print(comparison.to_string(index=False))

### Exercise 5.2 — Visualize comparison

Create a horizontal bar chart comparing the test MAE of all models.

**Why:** Visualizations make differences between models immediately apparent.

*Hint:* `plt.barh(...)`, color the best model differently.

**Question:** Which model performed best? Is the improvement over the baseline significant?

In [ ]:
plt.figure(figsize=(10, 5))
colors = ['green' if i == 0 else 'steelblue' for i in range(len(comparison))]
plt.barh(comparison['Model'], comparison['Test MAE'], color=colors, edgecolor='k')
plt.xlabel('Test MAE')
plt.title('Model Comparison — Test MAE on Diabetes Dataset')
for i, (_, row) in enumerate(comparison.iterrows()):
    plt.text(row['Test MAE'] + 0.3, i, f"{row['Test MAE']:.2f}", va='center')
plt.tight_layout()
plt.show()

**Answer:** On this dataset, linear regression is competitive because the features were designed to have linear relationships. Non-linear models may provide modest improvements, but the key takeaway is that the simplest model sometimes wins — the important thing is to compare systematically.


---
## 7. Hyperparameter Tuning with GridSearchCV

So far, we tried a few hyperparameter values by hand. **GridSearchCV** automates this process:

1. Define a **grid** of hyperparameter combinations
2. For each combination, run **K-fold cross-validation** on the training set
3. Select the combination with the best average CV score
4. Evaluate the best model on the test set

This is more systematic and less error-prone than manual tuning.


### Exercise 6.1 — GridSearchCV on Gradient Boosting

Define a parameter grid and run GridSearchCV with 5-fold cross-validation.

**Why:** Systematic search finds the best combination of hyperparameters, not just the best value for each individually.

*Hint:*
```python
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 5],
}
GridSearchCV(..., cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
```

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 5],
}

grid_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)
grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV MAE: {-grid_search.best_score_:.2f}")

### Exercise 6.2 — Evaluate tuned model on test set

Use `grid_search.best_estimator_` to predict on the test set. Compare with the baseline.

**Why:** The test set gives the final, unbiased performance estimate. Compare with baseline to see if tuning helped.

*Hint:* `grid_search.best_estimator_.predict(X_test_scaled)`


In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)

tuned_mae = mean_absolute_error(y_test, y_pred_tuned)

print("=== Tuned Gradient Boosting ===")
print(f"Test MAE: {tuned_mae:.2f}")

print(f"\n=== Improvement over baseline ===")
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Tuned MAE:    {tuned_mae:.2f}")
print(f"Improvement:  {baseline_mae - tuned_mae:+.2f}")

---
## 8. Cross-Validation

A single train/test split depends on the random seed. Cross-validation gives a more **robust** performance estimate by evaluating the model across multiple folds.


### Exercise 7.1 — 5-fold cross-validation on best model

Run 5-fold cross-validation on the full dataset using the best model from GridSearchCV.

**Why:** Cross-validation tells us how stable the model is across different data splits.

*Hint:* `cross_val_score(grid_search.best_estimator_, scaler.transform(X), y, cv=5, scoring='neg_mean_absolute_error')`

**Question:** Is the model stable across folds? How does the CV score compare to the test set score?

In [ ]:
cv_scores = cross_val_score(
    grid_search.best_estimator_,
    scaler.transform(X), y,
    cv=5, scoring='neg_mean_absolute_error'
)

cv_mae = -cv_scores
print(f"5-Fold CV MAE scores: {cv_mae}")
print(f"Mean MAE: {cv_mae.mean():.2f} ± {cv_mae.std():.2f}")

**Answer:** The CV score gives a more robust estimate than a single train/test split. If the CV MAE is close to the test MAE, our evaluation is reliable.

---
## 9. Summary & Key Takeaways

### What We Learned

1. **Always establish a baseline** before trying complex models
2. **Try multiple models** with different hyperparameters
3. **Use GridSearchCV** for systematic hyperparameter optimization
4. **Compare models on the SAME test set** for fair comparison
5. **Cross-validate** for robust performance estimates
6. **Simple models sometimes win** — complexity isn't always better

### Workflow Summary

| Step | What you did |
|------|-------------|
| **Baseline** | Linear Regression |
| **Manual exploration** | KNN, Decision Tree, SVR, Random Forest, Gradient Boosting |
| **Systematic tuning** | GridSearchCV on Gradient Boosting |
| **Validation** | 5-fold cross-validation |

### Key Principle

**Compare systematically.** The best model depends on the dataset — always test multiple approaches and let the data decide.
